# 01 — Preparación de Datos
## Amazon Fine Food Reviews
### Grupo 6
Este cuaderno carga los datos crudos, los limpia y exporta
un archivo procesado listo para el EDA.

### Integrantes
Jose Eduardo Arevalo Sarango,  
Monica Katherine Cholango Collaguazo,
Byron Fabricio Torres Apolo

In [1]:
!pip install kaggle -q

In [3]:
import os

os.environ['KAGGLE_USERNAME'] = 'monchis0731'
os.environ['KAGGLE_KEY'] = 'export KAGGLE_API_TOKEN=KGAT_60e50ef62be0eb3cc40f7f6207dc76f9'

print("Kaggle configurado correctamente")

Kaggle configurado correctamente


In [4]:
!pip install kaggle -q
!kaggle datasets download -d snap/amazon-fine-food-reviews
!unzip -q amazon-fine-food-reviews.zip
!ls -lh

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:01<00:00, 155MB/s]

total 885M
-rw-r--r-- 1 root root 243M Sep 19  2019 amazon-fine-food-reviews.zip
-rw-r--r-- 1 root root 356M Sep 19  2019 database.sqlite
-rw-r--r-- 1 root root  277 Sep 19  2019 hashes.txt
-rw-r--r-- 1 root root 287M Sep 19  2019 Reviews.csv
drwxr-xr-x 1 root root 4.0K May 12 13:29 sample_data


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [6]:
df = pd.read_csv('Reviews.csv')
print("Shape:", df.shape)
print("\nPrimeras filas:")
df.head()

Shape: (568454, 10)

Primeras filas:


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## 2. Filtro de calidad estadística
Solo conservamos reseñas con al menos 5 votos recibidos.
Reseñas con pocos votos producen tasas de utilidad poco representativas.

In [7]:
antes = df.shape[0]
df = df[df['HelpfulnessDenominator'] >= 5]
print(f"Filas antes: {antes:,}")
print(f"Filas después del filtro: {df.shape[0]:,}")
print(f"Filas eliminadas: {antes - df.shape[0]:,}")

Filas antes: 568,454
Filas después del filtro: 67,467
Filas eliminadas: 500,987


## 3. Eliminación de duplicados
Eliminamos duplicados por mismo usuario + mismo producto + misma fecha.

In [8]:
antes = df.shape[0]
df = df.drop_duplicates(subset=['UserId', 'ProductId', 'Time'])
print(f"Duplicados eliminados: {antes - df.shape[0]:,}")
print(f"Shape final: {df.shape}")

Duplicados eliminados: 485
Shape final: (66982, 10)


## 4. Conversión de fechas
Convertimos Time de Unix timestamp a datetime.

In [9]:
df['Time'] = pd.to_datetime(df['Time'], unit='s')
df['Year'] = df['Time'].dt.year
df['Month'] = df['Time'].dt.month
df['DayOfWeek'] = df['Time'].dt.dayofweek
print("Rango de fechas:")
print("Desde:", df['Time'].min())
print("Hasta:", df['Time'].max())

Rango de fechas:
Desde: 2000-01-19 00:00:00
Hasta: 2012-10-21 00:00:00


## 5. Variable objetivo
La variable objetivo es la utilidad percibida de la reseña.
- helpfulness_ratio >= 0.7 → útil (1)
- helpfulness_ratio < 0.7  → no útil (0)

In [10]:
df['helpfulness_ratio'] = df['HelpfulnessNumerator'] / df['HelpfulnessDenominator']

df['es_util'] = (df['helpfulness_ratio'] >= 0.7).astype(int)

print("Distribución de la variable objetivo:")
print(df['es_util'].value_counts())
print(f"\nPorcentaje útiles: {df['es_util'].mean()*100:.1f}%")

Distribución de la variable objetivo:
es_util
1    46610
0    20372
Name: count, dtype: int64

Porcentaje útiles: 69.6%


## 6. Features de texto
Creamos características derivadas del texto para el modelo.

In [11]:
# Longitud en palabras y número de oraciones
df['word_count'] = df['Text'].str.split().str.len()
df['sentence_count'] = df['Text'].str.count('[.!?]+')
df['summary_word_count'] = df['Summary'].str.split().str.len()

print("Features de texto creadas:")
print(df[['word_count', 'sentence_count', 'summary_word_count']].describe())

Features de texto creadas:
         word_count  sentence_count  summary_word_count
count  66982.000000    66982.000000        66957.000000
mean     113.184199        7.721448            4.522798
std      120.324929        7.487985            2.814448
min        3.000000        0.000000            1.000000
25%       44.000000        4.000000            2.000000
50%       78.000000        6.000000            4.000000
75%      138.000000        9.000000            6.000000
max     3432.000000      385.000000           30.000000


## 7. Score de sentimiento con VADER
VADER es un analizador de sentimiento basado en léxico,
no requiere entrenamiento y funciona bien con texto de reseñas.

In [14]:
import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Aplicamos VADER al texto (puede tardar 1-2 minutos)
df['vader_compound'] = df['Text'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

print("Score de sentimiento VADER:")
print(df['vader_compound'].describe())

Score de sentimiento VADER:
count    66982.000000
mean         0.526888
std          0.573598
min         -0.998500
25%          0.273200
50%          0.812600
75%          0.945300
max          0.999900
Name: vader_compound, dtype: float64


## 8. Coherencia entre sentimiento y estrellas
Verificamos si el sentimiento del texto es coherente con la calificación.
Reseña positiva (Score >= 4) con vader > 0 → coherente.
Reseña negativa (Score <= 2) con vader < 0 → coherente.

In [15]:
def es_coherente(row):
    if row['Score'] >= 4 and row['vader_compound'] > 0:
        return 1
    elif row['Score'] <= 2 and row['vader_compound'] < 0:
        return 1
    else:
        return 0

df['coherencia_sentimiento'] = df.apply(es_coherente, axis=1)
print("Coherencia sentimiento vs estrellas:")
print(df['coherencia_sentimiento'].value_counts())
print(f"\n% coherentes: {df['coherencia_sentimiento'].mean()*100:.1f}%")

Coherencia sentimiento vs estrellas:
coherencia_sentimiento
1    46163
0    20819
Name: count, dtype: int64

% coherentes: 68.9%


## 9. Validaciones finales

In [16]:
assert df.shape[0] > 0
assert 'es_util' in df.columns
assert 'vader_compound' in df.columns
assert 'coherencia_sentimiento' in df.columns
assert df['HelpfulnessDenominator'].min() >= 5

print("Todas las validaciones pasaron ✓")
print(f"Shape final: {df.shape}")
df.head()

Todas las validaciones pasaron ✓
Shape final: (66982, 20)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Year,Month,DayOfWeek,helpfulness_ratio,es_util,word_count,sentence_count,summary_word_count,vader_compound,coherencia_sentimiento
14,15,B001GVISJM,A2MUGFV2TDQ47K,"Lynrie ""Oh HELL no""",4,5,5,2010-03-12,Strawberry Twizzlers - Yummy,The Strawberry Twizzlers are my guilty pleasur...,2010,3,4,0.8,1,22,2,4.0,0.6486,1
15,16,B001GVISJM,A1CZX3CP8IKQIJ,Brian A. Lee,4,5,5,2009-12-29,"Lots of twizzlers, just what you expect.",My daughter loves twizzlers and this shipment ...,2009,12,1,0.8,1,24,3,7.0,0.5719,1
32,33,B001EO5QW8,AOVROBZ8BNTP7,S. Potter,19,19,4,2006-11-13,Best of the Instant Oatmeals,McCann's Instant Oatmeal is great if you must ...,2006,11,0,1.0,1,197,12,5.0,0.7103,1
33,34,B001EO5QW8,A3PMM0NFVEJGK9,"Megan ""Bad at Nicknames""",13,13,4,2006-12-17,Good Instant,This is a good instant oatmeal from the best o...,2006,12,6,1.0,1,90,5,2.0,0.9779,1
34,35,B001EO5QW8,A2EB6OGOWCRU5H,CorbyJames,9,9,5,2007-03-30,Great Irish oatmeal for those in a hurry!,Instant oatmeal can become soggy the minute th...,2007,3,4,1.0,1,94,4,8.0,0.9091,1


## 10. Exportar datos procesados

In [17]:
df.to_parquet('reviews_limpias.parquet', index=False)
print("Guardado: reviews_limpias.parquet")
print("Shape:", df.shape)

# Descargar para subir al repo
from google.colab import files
files.download('reviews_limpias.parquet')

Guardado: reviews_limpias.parquet
Shape: (66982, 20)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>